# Weld Defect Detection — YOLO Training & Evaluation Pipeline (GPU / Full Run)

Research-style pipeline for training and comparing YOLO object detection models on a weld defect
dataset exported from Label Studio (YOLO format). The dataset is already split into three
independent folders — `weld-train`, `weld-validation`, `weld-test` — which are used as-is and
never re-split or merged.

This is the **GPU-configured** variant of `train.ipynb`, tuned for a cloud/Colab-style A100 (80GB
VRAM, 12 vCPUs). `RUN_MODE = "full"` is now the active default — `smoke_test` is still fully
supported and selectable for a quick pipeline check on any machine.

**Notebook flow:** Run-mode config → Setup → Data preprocessing & EDA → Training → Evaluation →
Comparative analysis → Best model selection → Reproducibility notes.


## 0. Run Mode Config

This is the single control panel for the whole notebook. Every later section reads `RUN_MODE`
(and the other variables in this cell) and adapts automatically — you should never need to edit
code in other sections just to switch modes.

- **`full`** (**active default here** — GPU machine, A100 80GB): trains every model variant in
  `MODEL_VARIANTS` for `EPOCHS` epochs on the complete train/validation splits, then evaluates on
  the complete held-out test split. Hyperparameters below are tuned for the A100.
- **`smoke_test`** (still selectable): trains the smallest model for 1 epoch on a small random
  subsample of each split, purely to verify the pipeline runs end-to-end. Metrics produced in this
  mode are **not meaningful** — use it if you want to sanity-check the pipeline again before
  committing GPU time to a full run.


In [ ]:
# ============================================================
# RUN MODE CONFIG — the only cell you should need to edit to
# switch between a quick smoke test and a full training run.
# ============================================================

RUN_MODE = "full"  # options: "smoke_test" or "full" -- ACTIVE: full (A100 GPU run)

assert RUN_MODE in ("smoke_test", "full"), f"Unknown RUN_MODE: {RUN_MODE!r}"

# --- Dataset paths (already-split by Label Studio export; never re-split) ---
TRAIN_DIR = "weld-train"
VAL_DIR = "weld-validation"
TEST_DIR = "weld-test"

# --- Smoke-test subsampling (still available, e.g. RUN_MODE = "smoke_test") ---
SMOKE_TEST_SAMPLE_SIZE = 12  # images sampled independently from each of train/val/test
SMOKE_TEST_EPOCHS = 1
SMOKE_TEST_SEED = 42

# --- Full-mode training config ---
EPOCHS = 100  # real training run length; easy to edit

# --- Shared hyperparameters, tuned for an A100 80GB / 12 vCPU cloud instance ---
# (see Section 3 markdown for rationale)
IMG_SIZE = 640
BATCH_SIZE = 64      # A100 80GB comfortably fits a large batch at imgsz=640 for yolov8n/s/m.
                     # Set to -1 to use Ultralytics' auto-batch (fills VRAM to a target fraction)
                     # instead of this fixed value -- more robust if you change IMG_SIZE or add
                     # larger variants to MODEL_VARIANTS later.
AMP = True           # Automatic Mixed Precision -- A100 has strong fp16/bf16 tensor-core support.
NUM_WORKERS = 8      # dataloader workers; 12 vCPUs available, leave headroom for the main process.
RANDOM_SEED = 42

# --- Output locations ---
RUNS_DIR = "runs"
DATA_YAML_PATH = "data.yaml"
BEST_MODEL_EXPORT_PATH = "best_model.pt"

print(f"RUN_MODE = {RUN_MODE!r}")
if RUN_MODE == "smoke_test":
    print(f"  -> smoke test: {SMOKE_TEST_SAMPLE_SIZE} imgs/split, "
          f"{SMOKE_TEST_EPOCHS} epoch, smallest model only.")
    print("  -> Results in this mode are for pipeline verification only, NOT real performance.")
else:
    print(f"  -> FULL run: entire dataset, {EPOCHS} epochs, all model variants.")
    print(f"  -> batch={BATCH_SIZE}, imgsz={IMG_SIZE}, amp={AMP}, workers={NUM_WORKERS}")


In [ ]:
# --- GPU verification (this run is meant to use the A100; do NOT silently fall back to CPU) ---
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected, but this notebook is configured for a GPU run "
        "(A100 expected). Refusing to silently fall back to CPU.\n"
        "Check that you're connected to a GPU runtime/instance, then re-run this cell."
    )

DEVICE = "cuda"
gpu_index = 0
device_name = torch.cuda.get_device_name(gpu_index)
total_vram_gb = torch.cuda.get_device_properties(gpu_index).total_memory / (1024 ** 3)
cuda_version = torch.version.cuda
driver_version = torch.cuda.get_device_properties(gpu_index)  # placeholder, driver read below

print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"GPU name: {device_name}")
print(f"Total VRAM: {total_vram_gb:.1f} GB")
print(f"CUDA version (torch build): {cuda_version}")
try:
    import subprocess
    smi_driver = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]
    ).decode().strip()
    print(f"NVIDIA driver version: {smi_driver}")
except Exception as e:
    print(f"(Could not query driver version via nvidia-smi: {e})")

print(f"\nUsing device: {DEVICE}")


## 1. Setup

Install and import `ultralytics` (YOLO), `torch`, and supporting libraries. Random seeds are fixed
across `random`, `numpy`, and `torch` so that subsampling (smoke-test mode) and training are
reproducible run-to-run.


In [ ]:
# Install required packages (safe to re-run; pip no-ops if already satisfied)
import sys
%pip install -q ultralytics torch torchvision opencv-python-headless matplotlib pandas seaborn pyyaml pillow


In [ ]:
import os
import json
import random
import shutil
import time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import yaml
import torch
from PIL import Image
from ultralytics import YOLO

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)
print(f"Seeds set to {RANDOM_SEED}. Ultralytics version: {__import__('ultralytics').__version__}")
print(f"Torch version: {torch.__version__}")


## 2. Data Preprocessing & EDA

Each of the three folders (`weld-train`, `weld-validation`, `weld-test`) is loaded **independently**
— they are already the correct split and are never merged or re-shuffled together. For each folder we:

1. Parse its class file (`classes.txt`/`notes.json`) dynamically — class names/count are never
   hardcoded — and verify all three folders agree on the same class list.
2. Validate that every image has a matching label file (and vice versa), reporting orphan counts.
3. Plot class distribution per split to check for imbalance.
4. Report image counts / average size and show a sample grid with bounding boxes drawn.
5. In `smoke_test` mode, subsample each folder independently (fixed seed, configurable size). In
   `full` mode (active here) this is a no-op and the complete folders are used untouched.
6. Generate `data.yaml` pointing Ultralytics at the three folder paths directly.


### 2.1 Class file parsing

Supports either a `classes.txt` (one class name per line, line index = class id) or a
`notes.json` (Label Studio export, `categories: [{id, name}, ...]`). If both exist we prefer
`notes.json` since it explicitly encodes the id, then cross-check against `classes.txt`.


In [ ]:
def parse_classes(folder: str):
    """Return {class_id: class_name} for a YOLO-export folder, from notes.json and/or classes.txt."""
    folder = Path(folder)
    notes_path = folder / "notes.json"
    classes_txt_path = folder / "classes.txt"

    id_to_name_notes = None
    if notes_path.exists():
        with open(notes_path) as f:
            notes = json.load(f)
        cats = notes.get("categories", [])
        id_to_name_notes = {c["id"]: c["name"] for c in cats}

    id_to_name_txt = None
    if classes_txt_path.exists():
        with open(classes_txt_path) as f:
            names = [line.strip() for line in f if line.strip()]
        id_to_name_txt = {i: n for i, n in enumerate(names)}

    if id_to_name_notes is None and id_to_name_txt is None:
        raise FileNotFoundError(
            f"No classes.txt or notes.json found in {folder}. Cannot determine class list."
        )

    if id_to_name_notes is not None and id_to_name_txt is not None:
        if id_to_name_notes != id_to_name_txt:
            raise ValueError(
                f"Class mismatch in {folder}: notes.json={id_to_name_notes} "
                f"vs classes.txt={id_to_name_txt}"
            )
        return id_to_name_notes

    return id_to_name_notes if id_to_name_notes is not None else id_to_name_txt


def require_dir(folder: str, label: str):
    p = Path(folder)
    if not p.exists() or not p.is_dir():
        raise FileNotFoundError(f"{label} folder not found: {folder!r}. Check the path and re-run.")
    images_dir, labels_dir = p / "images", p / "labels"
    if not images_dir.exists():
        raise FileNotFoundError(f"{label}: missing images/ subfolder at {images_dir}")
    if not labels_dir.exists():
        raise FileNotFoundError(f"{label}: missing labels/ subfolder at {labels_dir}")
    return p


for _dir, _label in [(TRAIN_DIR, "TRAIN_DIR"), (VAL_DIR, "VAL_DIR"), (TEST_DIR, "TEST_DIR")]:
    require_dir(_dir, _label)

classes_train = parse_classes(TRAIN_DIR)
classes_val = parse_classes(VAL_DIR)
classes_test = parse_classes(TEST_DIR)

if not (classes_train == classes_val == classes_test):
    raise ValueError(
        "Class definitions disagree across splits!\n"
        f"train: {classes_train}\nval: {classes_val}\ntest: {classes_test}\n"
        "Fix the source export before proceeding."
    )

CLASS_MAP = dict(sorted(classes_train.items()))
CLASS_NAMES = [CLASS_MAP[i] for i in sorted(CLASS_MAP)]
NUM_CLASSES = len(CLASS_NAMES)

print(f"Detected {NUM_CLASSES} classes (agree across all 3 splits):")
for cid, name in CLASS_MAP.items():
    print(f"  {cid}: {name}")


### 2.2 Image/label integrity check

Flags any image without a matching label file, or any label file without a matching image
("orphans"), per folder.


In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def list_stems(folder: Path, exts):
    return {p.stem: p for p in folder.iterdir() if p.suffix.lower() in exts}

def check_integrity(folder: str, label: str):
    folder = Path(folder)
    images = list_stems(folder / "images", IMG_EXTS)
    labels = list_stems(folder / "labels", {".txt"})

    img_stems, lbl_stems = set(images), set(labels)
    orphan_images = img_stems - lbl_stems
    orphan_labels = lbl_stems - img_stems
    matched = img_stems & lbl_stems

    print(f"[{label}] images={len(images)}  labels={len(labels)}  matched={len(matched)}  "
          f"orphan_images={len(orphan_images)}  orphan_labels={len(orphan_labels)}")
    if orphan_images:
        print(f"  e.g. images with no label: {sorted(list(orphan_images))[:5]}")
    if orphan_labels:
        print(f"  e.g. labels with no image: {sorted(list(orphan_labels))[:5]}")

    return {
        "images": images, "labels": labels,
        "n_images": len(images), "n_labels": len(labels),
        "n_matched": len(matched),
        "orphan_images": orphan_images, "orphan_labels": orphan_labels,
    }

integrity = {}
for _dir, _label in [(TRAIN_DIR, "train"), (VAL_DIR, "validation"), (TEST_DIR, "test")]:
    integrity[_label] = check_integrity(_dir, _label)


### 2.3 Class distribution per split

Counts label instances (not images) per class, per split, to reveal imbalance both within a
split and across splits.


In [ ]:
def class_distribution(folder: str):
    folder = Path(folder)
    counts = Counter()
    for label_file in (folder / "labels").glob("*.txt"):
        with open(label_file) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                cid = int(line.split()[0])
                counts[cid] += 1
    return counts

dist = {
    "train": class_distribution(TRAIN_DIR),
    "validation": class_distribution(VAL_DIR),
    "test": class_distribution(TEST_DIR),
}

dist_df = pd.DataFrame(
    {split: [d.get(cid, 0) for cid in sorted(CLASS_MAP)] for split, d in dist.items()},
    index=CLASS_NAMES,
)
display(dist_df)

fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES * 1.1), 5))
dist_df.plot(kind="bar", ax=ax)
ax.set_title("Class distribution (instance counts) per split")
ax.set_xlabel("Class")
ax.set_ylabel("Instance count")
ax.legend(title="Split")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### 2.4 Dataset summary & sample images with bounding boxes

Reports image count and average image size for each split, then draws a grid of sample images
from `weld-train` with their ground-truth boxes overlaid, trying to cover as many distinct
classes as are actually present.


In [ ]:
def image_size_stats(folder: str, sample_n: int = 200):
    folder = Path(folder)
    files = list((folder / "images").iterdir())
    sample = random.sample(files, min(sample_n, len(files))) if files else []
    sizes = []
    for f in sample:
        try:
            with Image.open(f) as im:
                sizes.append(im.size)
        except Exception:
            continue
    if not sizes:
        return None
    widths, heights = zip(*sizes)
    return {"n_sampled": len(sizes), "avg_w": np.mean(widths), "avg_h": np.mean(heights)}

for _dir, _label in [(TRAIN_DIR, "train"), (VAL_DIR, "validation"), (TEST_DIR, "test")]:
    n_imgs = integrity[_label]["n_images"]
    stats = image_size_stats(_dir)
    if stats:
        print(f"[{_label}] {n_imgs} images | avg size (sampled {stats['n_sampled']}): "
              f"{stats['avg_w']:.0f} x {stats['avg_h']:.0f} px")
    else:
        print(f"[{_label}] {n_imgs} images | no images found to sample sizes from")


In [ ]:
def yolo_txt_to_boxes(label_path: Path, img_w: int, img_h: int):
    boxes = []
    if not label_path.exists():
        return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cid, xc, yc, w, h = int(parts[0]), *map(float, parts[1:5])
            x1 = (xc - w / 2) * img_w
            y1 = (yc - h / 2) * img_h
            x2 = (xc + w / 2) * img_w
            y2 = (yc + h / 2) * img_h
            boxes.append((cid, x1, y1, x2, y2))
    return boxes


def pick_diverse_samples(folder: str, n: int):
    """Greedily pick up to n images from `folder` covering as many distinct classes as possible."""
    folder = Path(folder)
    images_dir, labels_dir = folder / "images", folder / "labels"
    covered, picks = set(), []
    label_files = list(labels_dir.glob("*.txt"))
    random.shuffle(label_files)

    for lf in label_files:
        if len(picks) >= n:
            break
        with open(lf) as f:
            cids = {int(line.split()[0]) for line in f if line.strip()}
        if not cids or cids.issubset(covered):
            continue
        img_candidates = [p for p in images_dir.glob(f"{lf.stem}.*") if p.suffix.lower() in IMG_EXTS]
        if not img_candidates:
            continue
        picks.append(img_candidates[0])
        covered |= cids

    # top up with random remaining images if we still have room
    if len(picks) < n:
        remaining = [p for p in images_dir.iterdir() if p.suffix.lower() in IMG_EXTS and p not in picks]
        random.shuffle(remaining)
        picks.extend(remaining[: n - len(picks)])

    return picks[:n]


sample_imgs = pick_diverse_samples(TRAIN_DIR, n=9)
n_cols = 3
n_rows = int(np.ceil(len(sample_imgs) / n_cols)) if sample_imgs else 0

if sample_imgs:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    axes = np.array(axes).reshape(-1)
    palette = sns.color_palette("husl", NUM_CLASSES)

    for ax, img_path in zip(axes, sample_imgs):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        label_path = Path(TRAIN_DIR) / "labels" / f"{img_path.stem}.txt"
        boxes = yolo_txt_to_boxes(label_path, w, h)

        ax.imshow(img)
        for cid, x1, y1, x2, y2 in boxes:
            color = palette[cid % NUM_CLASSES]
            rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                  edgecolor=color, linewidth=2)
            ax.add_patch(rect)
            ax.text(x1, max(y1 - 5, 0), CLASS_MAP.get(cid, str(cid)),
                     color="white", fontsize=8,
                     bbox=dict(facecolor=color, alpha=0.7, pad=1))
        ax.set_title(img_path.name, fontsize=8)
        ax.axis("off")

    for ax in axes[len(sample_imgs):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No sample images found in TRAIN_DIR to display.")


### 2.5 Smoke-test subsampling (no-op in `full` mode)

In `smoke_test` mode, each of the three folders is independently subsampled to
`SMOKE_TEST_SAMPLE_SIZE` images (fixed seed) and copied into a parallel `*_smoke` directory tree
so the originals are left untouched. In `full` mode (active here) this cell is a no-op and the
complete, original `weld-train` / `weld-validation` / `weld-test` directories are used directly
everywhere below — no subsampling.


In [ ]:
def make_smoke_subset(src_folder: str, n: int, seed: int) -> str:
    src = Path(src_folder)
    dst = Path(f"{src_folder}_smoke")
    if dst.exists():
        shutil.rmtree(dst)
    (dst / "images").mkdir(parents=True)
    (dst / "labels").mkdir(parents=True)

    for f in ["classes.txt", "notes.json"]:
        if (src / f).exists():
            shutil.copy(src / f, dst / f)

    images = [p for p in (src / "images").iterdir() if p.suffix.lower() in IMG_EXTS]
    rng = random.Random(seed)
    sample = rng.sample(images, min(n, len(images)))

    for img_path in sample:
        shutil.copy(img_path, dst / "images" / img_path.name)
        label_path = src / "labels" / f"{img_path.stem}.txt"
        if label_path.exists():
            shutil.copy(label_path, dst / "labels" / label_path.name)

    return str(dst)


if RUN_MODE == "smoke_test":
    ACTIVE_TRAIN_DIR = make_smoke_subset(TRAIN_DIR, SMOKE_TEST_SAMPLE_SIZE, SMOKE_TEST_SEED)
    ACTIVE_VAL_DIR = make_smoke_subset(VAL_DIR, SMOKE_TEST_SAMPLE_SIZE, SMOKE_TEST_SEED)
    ACTIVE_TEST_DIR = make_smoke_subset(TEST_DIR, SMOKE_TEST_SAMPLE_SIZE, SMOKE_TEST_SEED)
    print("SMOKE TEST MODE: using subsampled folders below (not real performance numbers):")
else:
    ACTIVE_TRAIN_DIR, ACTIVE_VAL_DIR, ACTIVE_TEST_DIR = TRAIN_DIR, VAL_DIR, TEST_DIR
    print("FULL MODE: using complete dataset folders (no subsampling):")

for _label, _dir in [("train", ACTIVE_TRAIN_DIR), ("validation", ACTIVE_VAL_DIR), ("test", ACTIVE_TEST_DIR)]:
    n = len(list((Path(_dir) / "images").iterdir()))
    print(f"  {_label}: {_dir}  ({n} images)")

if RUN_MODE == "full":
    assert ACTIVE_TRAIN_DIR == TRAIN_DIR and ACTIVE_VAL_DIR == VAL_DIR and ACTIVE_TEST_DIR == TEST_DIR, (
        "full mode must point directly at the original, complete dataset folders."
    )


### 2.6 Generate `data.yaml`

Ultralytics needs a YAML config pointing at the three split directories, with `nc`/`names`
populated from the dynamically-parsed class map (never hardcoded).


In [ ]:
data_yaml = {
    "train": str(Path(ACTIVE_TRAIN_DIR).resolve() / "images"),
    "val": str(Path(ACTIVE_VAL_DIR).resolve() / "images"),
    "test": str(Path(ACTIVE_TEST_DIR).resolve() / "images"),
    "nc": NUM_CLASSES,
    "names": CLASS_NAMES,
}

with open(DATA_YAML_PATH, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"Wrote {DATA_YAML_PATH}:")
print(yaml.dump(data_yaml, sort_keys=False))


## 3. Training

We compare several YOLO model-size variants. In `smoke_test` mode only the smallest variant is
trained, for `SMOKE_TEST_EPOCHS` epoch(s), purely to confirm the training loop runs end-to-end.
In `full` mode (active here) **every** variant listed in `MODEL_VARIANTS` is trained for `EPOCHS`
epochs, each as its own run with a distinct name, on the complete train/validation data.

**Hyperparameters:** image size (`IMG_SIZE`) and batch size (`BATCH_SIZE`) are held constant
across all runs (set in Section 0) so that the comparison in Section 5 isolates the effect of
model size rather than confounding it with different training settings. On this A100 80GB
instance, `BATCH_SIZE=64` at `imgsz=640` leaves comfortable VRAM headroom for all three variants;
`AMP=True` enables mixed-precision training (fp16/bf16 tensor cores), which the A100 supports well
and which meaningfully speeds up training with negligible accuracy impact; `NUM_WORKERS=8`
dataloader workers use most of the 12 available vCPUs while leaving headroom for the main process.

This section is modular: re-running it only re-trains models and does not touch Section 2's data
prep or Section 4/5's evaluation code, so you can re-run evaluation/comparison independently
against already-trained weights.


In [ ]:
# Model variants to compare (smallest first). In smoke_test mode only variants[0] is used;
# in full mode (active here) ALL variants below are trained, each as its own run.
MODEL_VARIANTS = ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]

if RUN_MODE == "smoke_test":
    active_variants = [MODEL_VARIANTS[0]]
    active_epochs = SMOKE_TEST_EPOCHS
else:
    active_variants = MODEL_VARIANTS
    active_epochs = EPOCHS

print(f"Training variants: {active_variants}")
print(f"Epochs: {active_epochs}")
print(f"Shared hyperparameters: imgsz={IMG_SIZE}, batch={BATCH_SIZE}, amp={AMP}, "
      f"workers={NUM_WORKERS}, device={DEVICE}")


### 3.0 Pre-flight sanity check

Final confirmation before committing GPU time to the full training loop: run mode, dataset sizes
actually being trained on, model variants, epochs, and batch size. Read through this output and
make sure it matches what you expect before letting the next cell run.


In [ ]:
print("=" * 60)
print("PRE-FLIGHT SANITY CHECK -- review before starting training")
print("=" * 60)
print(f"RUN_MODE: {RUN_MODE}")
print(f"Device: {DEVICE}" + (f" ({device_name})" if DEVICE == "cuda" else ""))
print()
print("Dataset (images actually used this run):")
for _label, _dir in [("train", ACTIVE_TRAIN_DIR), ("validation", ACTIVE_VAL_DIR), ("test", ACTIVE_TEST_DIR)]:
    n = len(list((Path(_dir) / "images").iterdir()))
    print(f"  {_label:<12s}: {_dir:<20s} {n} images")
print()
print(f"Model variants to train: {active_variants}")
print(f"Epochs per variant: {active_epochs}")
print(f"Batch size: {BATCH_SIZE}{' (auto-batch)' if BATCH_SIZE == -1 else ''}")
print(f"Image size: {IMG_SIZE}")
print(f"AMP (mixed precision): {AMP}")
print(f"Dataloader workers: {NUM_WORKERS}")
print(f"Number of classes: {NUM_CLASSES}")
print("=" * 60)

if RUN_MODE == "smoke_test":
    print("NOTE: RUN_MODE='smoke_test' -- this is a quick pipeline check, not a real training run.")
else:
    print("Confirm the above looks correct, then run the next cell to start full training.")


In [ ]:
def run_name_for(variant: str) -> str:
    stem = Path(variant).stem  # e.g. "yolov8n"
    suffix = "smoke" if RUN_MODE == "smoke_test" else "full"
    return f"{stem}_{suffix}"


def train_variant(variant: str):
    run_name = run_name_for(variant)
    model = YOLO(variant)

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    results = model.train(
        data=DATA_YAML_PATH,
        epochs=active_epochs,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        amp=AMP,
        workers=NUM_WORKERS,
        device=DEVICE,
        seed=RANDOM_SEED,
        project=RUNS_DIR,
        name=run_name,
        exist_ok=True,
        verbose=True,
    )

    elapsed_sec = time.time() - start_time
    peak_mem_gb = (
        torch.cuda.max_memory_allocated() / (1024 ** 3) if DEVICE == "cuda" else float("nan")
    )

    print(f"\n[{run_name}] training time: {elapsed_sec / 60:.1f} min "
          f"({elapsed_sec:.0f} s) | peak GPU memory: {peak_mem_gb:.2f} GB")

    # Derive the weights path from the actual save_dir Ultralytics reports rather than
    # reconstructing it from project/name, since Ultralytics may nest the save_dir
    # differently depending on global settings (e.g. runs/detect/<project>/<name>).
    save_dir = Path(results.save_dir)
    return {
        "variant": variant,
        "run_name": run_name,
        "weights_best": str(save_dir / "weights" / "best.pt"),
        "save_dir": str(save_dir),
        "train_time_sec": elapsed_sec,
        "peak_gpu_mem_gb": peak_mem_gb,
    }


training_runs = {}
for variant in active_variants:
    print(f"\n=== Training {variant} (run: {run_name_for(variant)}) ===")
    training_runs[variant] = train_variant(variant)

print("\nTraining complete for:", list(training_runs.keys()))


### 3.1 Runtime / resource summary

Total wall-clock training time and peak GPU memory usage per run, captured during training above
-- useful for documenting compute cost in a research writeup.


In [ ]:
resource_rows = [
    {
        "model": variant,
        "run_name": info["run_name"],
        "train_time_min": info["train_time_sec"] / 60,
        "peak_gpu_mem_GB": info["peak_gpu_mem_gb"],
    }
    for variant, info in training_runs.items()
]
resource_df = pd.DataFrame(resource_rows)
display(resource_df)


### 3.2 Training curves

Loss and mAP curves are read from each run's `results.csv` (written automatically by Ultralytics)
and plotted per run.


In [ ]:
def plot_training_curves(run_info: dict):
    csv_path = Path(run_info["save_dir"]) / "results.csv"
    if not csv_path.exists():
        print(f"No results.csv found for {run_info['run_name']} at {csv_path}")
        return
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    loss_cols = [c for c in df.columns if "loss" in c]
    for c in loss_cols:
        axes[0].plot(df["epoch"], df[c], label=c)
    axes[0].set_title(f"{run_info['run_name']} — loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend(fontsize=7)

    map50_col = next((c for c in df.columns if "mAP50" in c and "95" not in c), None)
    if map50_col:
        axes[1].plot(df["epoch"], df[map50_col])
    axes[1].set_title(f"{run_info['run_name']} — mAP@0.5")
    axes[1].set_xlabel("epoch")

    map5095_col = next((c for c in df.columns if "mAP50-95" in c), None)
    if map5095_col:
        axes[2].plot(df["epoch"], df[map5095_col])
    axes[2].set_title(f"{run_info['run_name']} — mAP@0.5:0.95")
    axes[2].set_xlabel("epoch")

    plt.tight_layout()
    plt.show()


for variant, run_info in training_runs.items():
    plot_training_curves(run_info)


## 4. Evaluation

Every trained model is evaluated on `weld-test` (or its smoke-test subset) — the held-out split
that is never touched during training or validation. For each model we report per-class and
overall precision/recall/F1/mAP@0.5/mAP@0.5:0.95, a confusion matrix, and qualitative predictions
including at least one failure case where available.

This section only needs `training_runs` (from Section 3) to exist — it can be re-run on its own
without retraining, as long as the weights files referenced in `training_runs` still exist on disk.


In [ ]:
def evaluate_variant(run_info: dict):
    model = YOLO(run_info["weights_best"])
    metrics = model.val(
        data=DATA_YAML_PATH,
        split="test",
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        verbose=False,
    )
    return model, metrics


eval_results = {}
for variant, run_info in training_runs.items():
    print(f"Evaluating {variant} on weld-test...")
    model, metrics = evaluate_variant(run_info)
    eval_results[variant] = {"model": model, "metrics": metrics, "run_info": run_info}


In [ ]:
def per_class_report(variant: str, metrics):
    names = metrics.names  # {id: name} as seen by this model
    p, r, f1_arr = metrics.box.p, metrics.box.r, metrics.box.f1
    ap50, ap5095 = metrics.box.ap50, metrics.box.ap

    rows = []
    present_class_ids = sorted(names.keys())
    for i, cid in enumerate(present_class_ids):
        rows.append({
            "class": names[cid],
            "precision": p[i] if i < len(p) else float("nan"),
            "recall": r[i] if i < len(r) else float("nan"),
            "f1": f1_arr[i] if i < len(f1_arr) else float("nan"),
            "mAP50": ap50[i] if i < len(ap50) else float("nan"),
            "mAP50-95": ap5095[i] if i < len(ap5095) else float("nan"),
        })
    df = pd.DataFrame(rows)

    overall = {
        "class": "ALL (mean)",
        "precision": metrics.box.mp,
        "recall": metrics.box.mr,
        "f1": np.nanmean(f1_arr) if len(f1_arr) else float("nan"),
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
    }
    df = pd.concat([df, pd.DataFrame([overall])], ignore_index=True)
    print(f"\n=== {variant}: per-class test metrics ===")
    display(df)
    return df


per_class_reports = {}
for variant, res in eval_results.items():
    per_class_reports[variant] = per_class_report(variant, res["metrics"])


### 4.1 Confusion matrices

Ultralytics writes a confusion matrix plot into each run's validation save directory; we display
those here.


In [ ]:
for variant, res in eval_results.items():
    cm_path = Path(res["metrics"].save_dir) / "confusion_matrix.png"
    print(f"=== {variant} confusion matrix ===")
    if cm_path.exists():
        img = plt.imread(cm_path)
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.axis("off")
        plt.title(variant)
        plt.show()
    else:
        print(f"  (no confusion matrix found at {cm_path})")


### 4.2 Qualitative predictions (predicted vs. ground truth)

For each model, a handful of `weld-test` images are shown with predicted boxes (solid) and
ground-truth boxes (dashed) overlaid, including at least one clear failure case (a prediction
that disagrees with ground truth) if one exists among the sampled images.


In [ ]:
def iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    denom = areaA + areaB - inter
    return inter / denom if denom > 0 else 0.0


def is_failure_case(gt_boxes, pred_boxes, iou_thresh=0.5):
    matched_gt = set()
    for pcid, *pbox in pred_boxes:
        best_iou, best_j = 0.0, -1
        for j, (gcid, *gbox) in enumerate(gt_boxes):
            if gcid != pcid or j in matched_gt:
                continue
            cur_iou = iou(pbox, gbox)
            if cur_iou > best_iou:
                best_iou, best_j = cur_iou, j
        if best_iou >= iou_thresh:
            matched_gt.add(best_j)
        else:
            return True  # unmatched prediction -> false positive
    return len(matched_gt) < len(gt_boxes)  # missed a ground-truth box


def show_predictions(variant: str, model, test_dir: str, n: int = 4):
    images_dir = Path(test_dir) / "images"
    all_imgs = [p for p in images_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    if not all_imgs:
        print(f"No test images found in {test_dir}")
        return
    sample = random.sample(all_imgs, min(n, len(all_imgs)))

    fig, axes = plt.subplots(1, len(sample), figsize=(5 * len(sample), 5))
    axes = np.atleast_1d(axes)
    palette = sns.color_palette("husl", NUM_CLASSES)
    found_failure = False

    for ax, img_path in zip(axes, sample):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        gt_boxes = yolo_txt_to_boxes(Path(test_dir) / "labels" / f"{img_path.stem}.txt", w, h)

        pred = model.predict(str(img_path), device=DEVICE, verbose=False)[0]
        pred_boxes = [
            (int(cls), *box.tolist())
            for cls, box in zip(pred.boxes.cls, pred.boxes.xyxy)
        ]

        ax.imshow(img)
        for cid, x1, y1, x2, y2 in gt_boxes:
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                        edgecolor=palette[cid % NUM_CLASSES],
                                        linewidth=2, linestyle="--"))
        for cid, x1, y1, x2, y2 in pred_boxes:
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                        edgecolor=palette[cid % NUM_CLASSES],
                                        linewidth=2, linestyle="-"))

        failure = is_failure_case(gt_boxes, pred_boxes)
        found_failure = found_failure or failure
        ax.set_title(f"{img_path.name}\n{'FAILURE CASE' if failure else 'ok'}",
                     color="red" if failure else "black", fontsize=8)
        ax.axis("off")

    fig.suptitle(f"{variant} — dashed=ground truth, solid=prediction")
    plt.tight_layout()
    plt.show()

    if not found_failure:
        print(f"  (no failure case found among the {len(sample)} sampled images for {variant})")


for variant, res in eval_results.items():
    show_predictions(variant, res["model"], ACTIVE_TEST_DIR, n=4)


## 5. Comparative Analysis

A summary table and charts comparing all trained models on accuracy metrics, inference speed,
model size, and training compute cost.

> **Smoke-test caveat:** if `RUN_MODE` is switched back to `smoke_test`, this table will have only
> one row and the metrics will not be meaningful. With `RUN_MODE = "full"` (active here) all
> `MODEL_VARIANTS` are trained and compared for real.


In [ ]:
def measure_inference_speed(model, test_dir: str, n: int = 20):
    images_dir = Path(test_dir) / "images"
    imgs = [p for p in images_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    sample = random.sample(imgs, min(n, len(imgs)))
    if not sample:
        return float("nan")

    # warmup
    model.predict(str(sample[0]), device=DEVICE, verbose=False)

    start = time.time()
    for img_path in sample:
        model.predict(str(img_path), device=DEVICE, verbose=False)
    elapsed = time.time() - start
    return (elapsed / len(sample)) * 1000  # ms/image


def model_size_mb(weights_path: str):
    return Path(weights_path).stat().st_size / (1024 ** 2)


summary_rows = []
for variant, res in eval_results.items():
    metrics = res["metrics"]
    model = res["model"]
    run_info = res["run_info"]
    n_params = sum(p.numel() for p in model.model.parameters())
    summary_rows.append({
        "model": variant,
        "run_name": run_info["run_name"],
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
        "precision": metrics.box.mp,
        "recall": metrics.box.mr,
        "f1": np.nanmean(metrics.box.f1) if len(metrics.box.f1) else float("nan"),
        "inference_ms_per_image": measure_inference_speed(model, ACTIVE_TEST_DIR),
        "model_size_MB": model_size_mb(run_info["weights_best"]),
        "params": n_params,
        "train_time_min": run_info["train_time_sec"] / 60,
        "peak_gpu_mem_GB": run_info["peak_gpu_mem_gb"],
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metric_cols = ["mAP50", "mAP50-95", "precision", "recall", "f1"]
summary_df.set_index("model")[metric_cols].plot(kind="bar", ax=axes[0])
axes[0].set_title("Accuracy metrics by model")
axes[0].set_ylabel("score")
axes[0].legend(fontsize=8)
plt.setp(axes[0].get_xticklabels(), rotation=45, ha="right")

ax2 = axes[1]
ax2.bar(summary_df["model"], summary_df["inference_ms_per_image"], color="steelblue", label="ms/image")
ax2.set_ylabel("Inference time (ms/image)", color="steelblue")
ax2.tick_params(axis="x", rotation=45)

ax3 = ax2.twinx()
ax3.plot(summary_df["model"], summary_df["model_size_MB"], color="darkorange", marker="o", label="size (MB)")
ax3.set_ylabel("Model size (MB)", color="darkorange")

axes[1].set_title("Speed vs. size by model")
plt.tight_layout()
plt.show()


### Tradeoff discussion

*(Fill in once this full run has completed on real data -- with `RUN_MODE = "full"` these numbers
are now real and can support a genuine writeup.)*

Expected shape of the tradeoff: smaller variants (e.g. `yolov8n`) should give lower latency, a
smaller footprint, and cheaper/faster training at the cost of lower mAP, especially on harder or
less-represented defect classes; larger variants (`yolov8m` and up) should trade inference speed,
model size, and training compute (see `train_time_min` / `peak_gpu_mem_GB` in the table above) for
higher accuracy. The best choice depends on the deployment constraint (e.g. real-time inspection
line vs. offline batch review) — see the selection criterion in Section 6.


## 6. Best Model Selection

**Selection criterion (edit this to change what "best" means):** the model with the highest
`mAP@0.5:0.95` on `weld-test` among those with inference time under `MAX_INFERENCE_MS`
milliseconds/image. If no model satisfies the latency constraint, we fall back to the model with
the highest `mAP@0.5:0.95` overall (and print a warning).


In [ ]:
MAX_INFERENCE_MS = 200  # latency budget in ms/image; edit as needed


def select_best_model(df: pd.DataFrame, max_inference_ms: float = MAX_INFERENCE_MS):
    eligible = df[df["inference_ms_per_image"] <= max_inference_ms]
    if eligible.empty:
        print(f"WARNING: no model met the {max_inference_ms} ms/image budget; "
              "falling back to best mAP50-95 overall.")
        eligible = df
    best_row = eligible.sort_values("mAP50-95", ascending=False).iloc[0]
    return best_row


best_row = select_best_model(summary_df)
best_variant = best_row["model"]

print("Selection criterion: highest mAP@0.5:0.95 on weld-test among models under "
      f"{MAX_INFERENCE_MS} ms/image (fallback: best mAP@0.5:0.95 overall).")
print(f"\nSelected best model: {best_variant}")
display(best_row.to_frame().T)


In [ ]:
best_weights_src = training_runs[best_variant]["weights_best"]
shutil.copy(best_weights_src, BEST_MODEL_EXPORT_PATH)
print(f"Copied best weights ({best_weights_src}) -> {BEST_MODEL_EXPORT_PATH}")


### Final summary report

**Selected model:** the variant chosen above by the stated criterion (highest mAP@0.5:0.95 on
`weld-test` under the inference-time budget).

**Rationale:** this criterion favors detection accuracy on the actual held-out test data while
enforcing a hard latency ceiling suitable for a practical inspection workflow; adjust
`MAX_INFERENCE_MS` (or replace `select_best_model` with a weighted-score function) if a different
tradeoff is preferred.

> **Reminder:** if `RUN_MODE = "smoke_test"`, the "selection" above is only a pipeline check —
> with a single model trained for 1 epoch on a handful of images there is nothing meaningful to
> select between yet. With `RUN_MODE = "full"` (active here), this selection reflects real
> performance across all `MODEL_VARIANTS`.


## 7. Reproducibility & Notes

Self-documenting summary for a research writeup: package versions, dataset sizes actually used,
detected classes, training compute cost, and assumptions made by this notebook.


In [ ]:
import ultralytics as _ultra

print("=== Reproducibility notes ===\n")

print(f"RUN_MODE: {RUN_MODE}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Device used: {DEVICE}" + (f" ({device_name}, {total_vram_gb:.1f} GB VRAM)" if DEVICE == "cuda" else ""))
print(f"Hyperparameters: imgsz={IMG_SIZE}, batch={BATCH_SIZE}, amp={AMP}, workers={NUM_WORKERS}")
print()

print("Package versions:")
print(f"  ultralytics: {_ultra.__version__}")
print(f"  torch: {torch.__version__}")
print(f"  numpy: {np.__version__}")
print(f"  pandas: {pd.__version__}")
print(f"  opencv: {cv2.__version__}")
print()

print("Dataset size actually used per folder:")
for label, real_dir, active_dir in [
    ("train", TRAIN_DIR, ACTIVE_TRAIN_DIR),
    ("validation", VAL_DIR, ACTIVE_VAL_DIR),
    ("test", TEST_DIR, ACTIVE_TEST_DIR),
]:
    real_n = integrity[label]["n_images"]
    used_n = len(list((Path(active_dir) / "images").iterdir()))
    tag = "full dataset" if RUN_MODE == "full" else f"smoke-test subsample (of {real_n})"
    print(f"  {label}: {used_n} images used ({tag})")
print()

print(f"Number of classes detected: {NUM_CLASSES}")
print(f"Classes: {CLASS_NAMES}")
print()

print("Training compute cost per model:")
for variant, info in training_runs.items():
    print(f"  {variant}: {info['train_time_sec'] / 60:.1f} min, "
          f"peak GPU memory {info['peak_gpu_mem_gb']:.2f} GB")
print()

print("Assumptions made by this notebook:")
print("  - weld-train / weld-validation / weld-test are already the correct, final split "
      "and are never merged or re-shuffled together.")
print("  - classes.txt / notes.json in each folder are authoritative and must agree "
      "across all three splits (verified in Section 2.1); the notebook errors out if they don't.")
print("  - YOLO label format: '<class_id> <x_center> <y_center> <width> <height>', all normalized [0,1].")
print("  - Model variants are downloaded automatically by ultralytics on first use "
      "(requires internet access on first run).")
print("  - This notebook hard-fails if no CUDA GPU is detected rather than silently "
      "falling back to CPU, since it is configured for a GPU (A100) full run.")
print("  - In smoke_test mode, metrics/plots throughout this notebook are for pipeline "
      "verification only and are not indicative of real model performance.")
